<!-- Cache bust 11_iterators_and_generators_notebook -->

# Iterators & Generators

***

### 🔹 1. Iterables vs Iterators
To understand loops and lazy evaluation, we must first understand the difference between **Iterables** and **Iterators**.

* **Iterable:** Any object that can return its members one at a time (e.g., list, tuple, string, dictionary). It implements the `__iter__()` method.
* **Iterator:** An object representing a stream of data. It yields one element at a time upon calling `__next__()`. It statefully remembers its current position.

#### 🧠 How a `for` loop works under the hood:
1. Calls `iter()` on the iterable to get an iterator.
2. Repeatedly calls `next()` on the iterator to retrieve elements.
3. Catches the `StopIteration` exception to exit the loop cleanly.

In [ ]:
numbers = [10, 20, 30]

# Get the iterator
num_iterator = iter(numbers)
print("Iterator object:", num_iterator)

# Fetch elements manually
print(next(num_iterator)) # 10
print(next(num_iterator)) # 20
print(next(num_iterator)) # 30

# The next call will raise StopIteration
try:
    print(next(num_iterator))
except StopIteration:
    print("Reached the end of the stream (StopIteration raised).")

***

### 🔹 2. Custom Iterators
You can create a custom iterator class by implementing the **Iterator Protocol**:
1. `__iter__(self)`: Must return the iterator object itself (typically `self`).
2. `__next__(self)`: Must return the next value in the stream or raise `StopIteration` when finished.

In [ ]:
class Counter:
    def __init__(self, low, high):
        self.current = low
        self.high = high
        
    def __iter__(self):
        return self
        
    def __next__(self):
        if self.current > self.high:
            raise StopIteration
        else:
            num = self.current
            self.current += 1
            return num

# Using our custom iterator in a for loop
for num in Counter(1, 4):
    print(num)

***

### 🔹 3. Generator Functions & yield
Creating custom iterator classes can be tedious and boilerplate-heavy. **Generators** provide a simple and elegant way to create iterators using functions.

* A generator function is defined like a normal function but uses the **`yield`** keyword instead of `return`.
* When a generator function is called, it returns a generator object without executing the function body.
* When `next()` is called, the function executes until it hits `yield`. It pauses, returns that value, and saves its execution state.
* The next `next()` call resumes execution exactly where it left off.

In [ ]:
def my_generator():
    print("First item execution")
    yield "Item A"
    print("Second item execution")
    yield "Item B"
    print("Third item execution")
    yield "Item C"

gen = my_generator()
print("Generator object:", gen)

# Fetching values
print(next(gen))
print(next(gen))
print(next(gen))

***

### 🔹 4. Memory Efficiency of Generators
* Standard lists store **all elements** in memory at once.
* Generators generate **one element at a time** on-demand (lazy evaluation).
* This is extremely critical in Data Science and ML for processing gigabytes of text, logs, or large image batches without running out of RAM (OOM - Out of Memory errors).

In [ ]:
import sys

# 1. List containing 1 million integers
list_data = [x for x in range(1000000)]
print(f"Memory used by List: {sys.getsizeof(list_data) / (1024*1024):.2f} MB")

# 2. Generator producing 1 million integers
gen_data = (x for x in range(1000000))
print(f"Memory used by Generator: {sys.getsizeof(gen_data):.2f} bytes")

***

### 🔹 5. Generator Expressions
Just like List Comprehensions create lists, **Generator Expressions** create generator objects.
* **Syntax:** Wrap the comprehension in parenthesis `()` instead of square brackets `[]`.

In [ ]:
# List Comprehension (Immediate evaluation)
squares_list = [x**2 for x in range(5)]

# Generator Expression (Lazy evaluation)
squares_gen = (x**2 for x in range(5))

print("List:", squares_list)
print("Generator:", squares_gen)

for val in squares_gen:
    print(val)

***

### 🔹 6. Advanced Generator Methods
Python generators can receive data back and manage sub-generators using advanced methods:

1. **`send(value)`**: Sends a value back into the generator. It becomes the result of the current `yield` expression.
2. **`close()`**: Stops the generator, raising `GeneratorExit`.
3. **`yield from`**: Delegates operations to another iterable or generator, keeping code clean.

In [ ]:
def running_average():
    total = 0
    count = 0
    average = None
    while True:
        # yield returns 'average', and pauses.
        # When .send(val) is called, 'val' is received here.
        val = yield average
        if val is None:
            break
        total += val
        count += 1
        average = total / count

avg_gen = running_average()
next(avg_gen) # Prime the generator (must run to the first yield)

print("Average after sending 10:", avg_gen.send(10))
print("Average after sending 20:", avg_gen.send(20))
print("Average after sending 30:", avg_gen.send(30))
avg_gen.close()

In [ ]:
def sub_gen():
    yield "Sub A"
    yield "Sub B"

def main_gen():
    yield "Main Start"
    yield from sub_gen() # Delegate to sub_gen
    yield "Main End"

for val in main_gen():
    print(val)

***

### 📝 Practice Questions

1. **Infinite Fibonacci Generator:** Write a generator function `fibonacci_gen()` that yields Fibonacci numbers infinitely (on-demand). Test it by printing the first 15 Fibonacci numbers.
2. **File Streaming Reader:** Write a generator function `read_large_file(file_path)` that yields one line at a time from a text file, stripping whitespace. This ensures we can read files of any size without loading the whole file into RAM.
3. **Running Product with send():** Write a generator that tracks the running product of all integers passed to it via `.send()`. Remember to prime it with a `next()` call first.